#This colab file will generate a RAG corpus to be used by Culinary agent.

First thing we are doing is going through flavordb. https://cosylab.iiitd.edu.in/flavordb.

This corpus will give us a mapping of type of food -> odorant. Odorants are the olafactory molecules that help humans taste things.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = "/content/drive/MyDrive/Culinary Agent/rag_docs"

ValueError: mount failed

In [ ]:
import requests
import json
from collections import defaultdict
from tqdm import tqdm
import os

# Using this URL, we can access flavorDB. We toggle the id from 0 to 972
# At each URL we get the profile of a unique ingredient in JSON format
# Which we then parse into ingredient -> odorant data
# We also create a document of odorant -> foods, so that we can easily search by odorant

BASE_URL = "https://cosylab.iiitd.edu.in/flavordb/entities_json?id={}"
MAX_ID = 972

os.makedirs(DATA_DIR, exist_ok=True)

food_to_odorants = {}
odorant_to_foods = defaultdict(lambda: {
    "foods": set(),
    "descriptors": set()
})

def parse_tags(s):
    if not s:
        return []
    return [x.strip().lower() for x in s.split("@") if x.strip()]

for i in tqdm(range(MAX_ID + 1)):

    try:
        r = requests.get(BASE_URL.format(i), timeout=10)
        if r.status_code != 200:
            continue

        data = r.json()
        food = data["entity_alias_readable"].lower()

        molecules = data.get("molecules", [])
        odorants = []

        for mol in molecules:

            # only keep odor-active compounds
            if mol.get("flavornet_id") != 1:
                continue

            name = mol.get("common_name") or mol.get("iupac_name")
            if not name:
                continue

            name = name.lower()

            descriptors = set()
            descriptors.update(parse_tags(mol.get("flavor_profile")))
            descriptors.update(parse_tags(mol.get("fooddb_flavor_profile")))
            descriptors.update(parse_tags(mol.get("odor")))

            odorant = {
                "name": name,
                "pubchem_id": mol.get("pubchem_id"),
                "descriptors": list(descriptors),
                "fema_profile": mol.get("fema_flavor_profile"),
                "functional_groups": parse_tags(mol.get("functional_groups"))
            }

            odorants.append(odorant)

            odorant_to_foods[name]["foods"].add(food)
            odorant_to_foods[name]["descriptors"].update(descriptors)

        if odorants:
            food_to_odorants[food] = odorants
            print(food)

    except Exception as e:
        print("Error", i, e)

# convert sets → lists
for o in odorant_to_foods:
    odorant_to_foods[o]["foods"] = list(odorant_to_foods[o]["foods"])
    odorant_to_foods[o]["descriptors"] = list(odorant_to_foods[o]["descriptors"])

# save
with open(f"{DATA_DIR}/food_to_odorants.json", "w") as f:
    json.dump(food_to_odorants, f, indent=2)

with open(f"{DATA_DIR}/odorants_to_foods.json", "w") as f:
    json.dump(odorant_to_foods, f, indent=2)

print("Corpus saved to Google Drive.")

In [ ]:
documents = []

for food, odorants in food_to_odorants.items():

    lines = [f"Ingredient: {food}\n"]

    for o in odorants[:10]:
        lines.append(
            f"Odorant: {o['name']} | descriptors: {', '.join(o['descriptors'])}"
        )

    documents.append({
        "ingredient": food,
        "text": "\n".join(lines)
    })

with open("flavor_rag_docs.json", "w") as f:
    json.dump(documents, f, indent=2)

In [ ]:
rag_docs = []

for food, odorants in food_to_odorant.items():

    odorant_names = [o["name"] for o in odorants]

    descriptors = []
    for o in odorants:
        descriptors.extend(o["descriptors"])

    text = f"""
Ingredient: {food}

Key odorants: {", ".join(odorant_names[:6])}

Flavor descriptors: {", ".join(set(descriptors)[:10])}

These aroma compounds define the characteristic flavor of {food}.
Foods that share these odorants often pair well in cooking.
"""

    rag_docs.append({
        "type": "ingredient_profile",
        "ingredient": food,
        "text": text
    })